In [2]:
import pandas as pd
import os
import textacy
from functools import partial
import spacy
import numpy as np
from fuzzywuzzy import fuzz
import itertools
from tqdm import tqdm

nlp = spacy.load("en_core_web_sm")

data_directory = os.getcwd()[:-4] + 'original_data/'

In [3]:
def clean_csv():
        df = pd.read_csv(data_directory + 'VideoInfo_ABC.csv')

        remove = ['#ABCNewsLiveUpdate', '#ABCNewsLivePrime', '#ABCNewsSpecial',
                '#ABCNEWSPRIME', '#ABCNewsPRIME', '#ABCNewsPride', '#ABCNewsPrime',
                '#ABCNewsLive', '#ABCNLPRIME','#ABCNLPRime', '#ABCNLPrime', 
                '#ABCNLUpdate', '#ABCNPRIME', '#ABNLPrime', '#ABCThisWeek', 
                '#ABC2020', '#ABCNEWS',  '#ABNews', '#ABCNEws','#ABCNL', 
                '#ABCNews','#ABCNewws','#ABCnews','#ABCNew', '#ABC','#WorldNewsTonight', 
                '#BreakingNews', '#Nightline', '#ThisWeek', '#TheBreakdown','#TheRundown', 
                'ThisWeek','ABC News Live Update: ','ABC News Prime: ', 'ABC News', 
                'ABC', 'Nightline', '|', 'WNT', 'GMA', 'BREAKING NEWS: ', "'", "`", ":", "’"]

        df['Title'] = df['Title'].str.replace(r'\|.*', '', regex=True)
        df['Title'] = df['Title'].str.replace(r' l ', ' ', regex=True)
        df['Description'] = df['Description'].apply(lambda x: x.splitlines()[0] if isinstance(x, str) else x)
        df['Description'] = df['Description'].str.replace(r'\#.*', '', regex=True)
        df['Text'] = df['Title']  + ' ' +  df['Description']

        for word in remove:
                df['Text'] = df['Text'].str.replace(word, '', regex=True)

        df['Text'] = df['Text'].apply(lambda x: str(x))

In [4]:
def get_ngrams(text):

    doc = nlp(text)

    unigrams = list(textacy.extract.ngrams(doc, n=1))
    bigrams = list(textacy.extract.ngrams(doc, n=2))
    trigrams = list(textacy.extract.ngrams(doc, n=3))
    quadgrams = list(textacy.extract.ngrams(doc, n=4))

    ngrams = unigrams + bigrams + trigrams + quadgrams

    ngrams = [' '.join([str(token) for token in ngram]) for ngram in ngrams]
    ngrams = [str(x) for x in ngrams]

    return ngrams

In [5]:
# df['ngrams'] = df['Text'].apply(lambda x : get_ngrams(x))

# df.to_csv(data_directory + 'ABCNews_Onkar_saved.csv', index=False)

# full_list = sum(df['ngrams'], [])

df = pd.read_csv(data_directory + 'ABCNews_Onkar_saved.csv')

In [6]:
#concatenate the ngrams column into a single list
full_list = list(itertools.chain.from_iterable(df['ngrams'].apply(lambda x: x.split(','))))
full_list = [x.replace('[','').replace(']','').replace("'",'').lstrip().rstrip().lower() for x in full_list]

In [7]:
a, count = np.unique(full_list, return_counts=True)

#sort a by count
a = a[np.argsort(count)[::-1]]
count = count[np.argsort(count)[::-1]]

good_ind = np.where(count >= 10)[0]

a = a[good_ind]

# a = a[:100]

In [8]:
#get all ngrams that consist of 2 or more words
two_more_ngram = [x for x in a if len(x.split()) > 1]
one_ngram = [x for x in a if len(x.split()) == 1]

len(two_more_ngram), len(one_ngram)

(2440, 5524)

In [9]:
# def partial_match(x,y):
#     return fuzz.token_set_ratio(x,y)
# partial_match_vector = np.vectorize(partial_match)


# combinations = list(itertools.product(two_more_ngram, a))
# combined_df = pd.DataFrame(combinations, columns=['List1', 'List2'])


# tqdm.pandas()
# combined_df['score']= combined_df.progress_apply(lambda x: partial_match_vector(x['List1'], x['List2']), axis=1)

# combined_df = combined_df[combined_df['List1'] != combined_df['List2']]

# combined_df.to_csv(data_directory + 'ABCNews_Onkar_fuzzy_score.csv', index=False)

In [10]:
combined_df = pd.read_csv(data_directory + 'ABCNews_Onkar_fuzzy_score.csv')

In [11]:
combined_df = combined_df[combined_df['score'] == 100]

#create a matrix based on the two lists, where the cell entry is the score and list1 and list2 are row and column names
matrix = combined_df.pivot(index='List1', columns='List2', values='score')

In [12]:
def get_sets(row):
    a = set(row.dropna().index) | set([row.name])
    return a

matrix['sets'] = matrix.apply(get_sets, axis=1)

In [13]:
def filter_largest_sets(sets):
    largest_sets = []
    
    for current_set in sets:
        is_subset = False
        for other_set in sets:
            if current_set != other_set and current_set.issubset(other_set):
                is_subset = True
                break
        if not is_subset:
            largest_sets.append(current_set)
    
    return largest_sets

final_final_sets = filter_largest_sets(matrix['sets'])

In [14]:
b = final_final_sets[10]

In [15]:
#count number of spaces in a string

words = [len(x.split()) for x in b]
lengths = np.array([len(x) for x in b])
smallest_ind = np.where(lengths == np.min(lengths))[0]

#get the lengths of smallest ind
small = np.argwhere(lengths == lengths[smallest_ind])[0][0]

b, small

({'$ 2', '$ 2 trillion', '2', '2 trillion', 'trillion'}, 4)

In [16]:
def find_smallest_word(strings):
    # Count the number of words in each string
    word_counts = [len(x.split()) for x in strings]
    
    # Find the minimum word count
    min_word_count = min(word_counts)
    
    # Get the strings with the minimum word count
    smallest_strings = [s for s, count in zip(strings, word_counts) if count == min_word_count]
    
    # Find the smallest word within those strings
    smallest_word = None
    for s in smallest_strings:
        words = s.split()
        for word in words:
            if smallest_word is None or len(word) < len(smallest_word):
                smallest_word = word
    
    return smallest_word

In [17]:
final_dict = {}
final_dict_replace = {}

for i, set_ in enumerate(final_final_sets):
    dic_key = find_smallest_word(set_)
    for word in set_:
        final_dict_replace[dic_key] = word


    dic_key = dic_key.strip().replace(' ', '-')
    for word in set_:
        final_dict[word] = dic_key

In [20]:
#use final_dict to replace the words in the text
def replace_words(text, final_dict):
    for key, value in final_dict.items():
        if isinstance(text, str):
            text = text.replace(key, value)
        else:
            pass
    return text


df['Text'] = df['Text'].apply(lambda x: replace_words(x, final_dict))

In [24]:
final_dict_replace.keys()

dict_keys(['1', '9', '10', '100', '12', '130', '15', '2', '20', '250', '5', '50', '500', '600', '000', '19', 's', '11', '13', '16', '17', '18', '1st', '3', '4', 'pt', '200', '2016', '2018', 'look', '2020', 'dnc', 'rnc', 'run', 'stay', '21', '22', '23', '24', '25', '25th', '27', '28', '2nd', '30', '41st', '48', '50th', '6', '60', '70', '75th', '911', 'abuse', 'awards', 'according', 'police', 'new', 'sexual', 'killing', 'active', 'john', 'act', 'african', 'ag', 'ahmaud', 'air', 'flight', 'senate', 'alex', 'vindman', 'attack', 'gunman', 'alleged', 'killed', 'shot', 'ukraine', 'amber', 'strong', 'airlines', 'history', 'people', 'woman', 'amid', 'dan', 'david', 'cuomo', 'yang', 'answers', 'anthony', 'brown', 'building', 'suicide', 'court', 'rating', 'blast', 'aretha', 'senator', 'army', 'articles', 'judd', 'asian', 'assault', 'charges', 'trial', 'general', 'jeff', 'barr', 'michael', 'rudy', 'said', 'missile', 'watch', 'bush', 'game', 'state', 'states', 'ben', 'best', 'beto', 'biden', 'win',

In [25]:
final_dict.keys()

dict_keys(['1 billion', '1', '1.9 trillion', '20/20 part 1', '1 year', '1.3', 'nearly 1', '1 watch', '1 million', '$ 1 billion', '$ 1.9', '$ 1 million', '1 dead', '1.9', '$ 1', '$ 1.9 trillion', '20/20 part 1 watch', 'billion', 'million', 'trillion', '9', '10 people', '$ 10', 'nearly 10', '10 a look', '10', '10 days', '10 years', '$ 100', '100 million', '100', 'nearly 100', '12 boys', '12 years', '12', '$ 12', '12 people', '130', '$ 130', '15', '15 a look', '15 years', '$ 15', '2 dead', 'day 2', '20/20 part 2', 'category 2', '2 trillion', '2 weeks', '2 million', '2 days', '$ 2', '20/20 part 2 watch', '2.5', 'night 2', 'nearly 2', '$ 2 trillion', '2 people', '2 watch', '2 a look', '2', '20/20 part 3', '20 years', '$ 20', '20', 'g-20', '20/20 part 5', '20/20', '20 million', 'episode of 20/20', '20/20 pt', 'nearly 20', 'bundy 20/20', '20/20 part 4', '20/20 part 4 watch', '20/20 part 3 watch', '20/20 mar', '20/20 part 5 watch', '250', '$ 250', '5', 'category 5', '5 watch', '$ 5', 'nearly 5

In [28]:
final_dict

{'1 billion': '1',
 '1': '1',
 '1.9 trillion': '9',
 '20/20 part 1': '1',
 '1 year': '1',
 '1.3': '1',
 'nearly 1': '1',
 '1 watch': '1',
 '1 million': '1',
 '$ 1 billion': '1',
 '$ 1.9': '9',
 '$ 1 million': '1',
 '1 dead': '1',
 '1.9': '9',
 '$ 1': '1',
 '$ 1.9 trillion': '9',
 '20/20 part 1 watch': '1',
 'billion': '1',
 'million': 'people',
 'trillion': '2',
 '9': '9',
 '10 people': '10',
 '$ 10': '10',
 'nearly 10': '10',
 '10 a look': '10',
 '10': '10',
 '10 days': '10',
 '10 years': '10',
 '$ 100': '100',
 '100 million': '100',
 '100': '100',
 'nearly 100': '100',
 '12 boys': '12',
 '12 years': '12',
 '12': '12',
 '$ 12': '12',
 '12 people': '12',
 '130': '130',
 '$ 130': '130',
 '15': '15',
 '15 a look': '15',
 '15 years': '15',
 '$ 15': '15',
 '2 dead': '2',
 'day 2': '2',
 '20/20 part 2': '2',
 'category 2': '2',
 '2 trillion': '2',
 '2 weeks': '2',
 '2 million': '2',
 '2 days': '2',
 '$ 2': '2',
 '20/20 part 2 watch': '2',
 '2.5': '5',
 'night 2': '2',
 'nearly 2': '2',
 '$ 

In [27]:
final_dict_replace

{'1': '$ 1',
 '9': 'look',
 '10': 'nearly $',
 '100': 'nearly 100',
 '12': 'years',
 '130': '$ 130',
 '15': 'years',
 '2': '$ 2',
 '20': '20',
 '250': '$ 250',
 '5': '$ 5',
 '50': '$ 50',
 '500': '500',
 '600': '$ 600',
 '000': 'votes',
 '19': 'tests positive',
 's': 'u.s.-mexico border',
 '11': 'apollo',
 '13': '13 a look',
 '16': 'look',
 '17': 'look',
 '18': '18',
 '1st': 'time',
 '3': 'nearly 3',
 '4': 'stage',
 'pt': '20',
 '200': 'people',
 '2016': 'interference in the 2016',
 '2018': 'year',
 'look': 'takes',
 '2020': '2020',
 'dnc': 'dnc',
 'rnc': 'rnc',
 'run': 'run',
 'stay': 'stay up to date',
 '21': '21',
 '22': '22 a look',
 '23': 'look',
 '24': '24',
 '25': 'years',
 '25th': 'amendment',
 '27': '27',
 '28': 'look',
 '2nd': 'trumps',
 '30': '30 years',
 '41st': 'president',
 '48': '48 hours',
 '50th': '50th',
 '6': 'jan.',
 '60': 'minutes',
 '70': '70 years',
 '75th': '75th',
 '911': '911',
 'abuse': 'sexual',
 'awards': 'academy',
 'according': 'according',
 'police': 'po